# MNIST Shared Generator

Trains one conditional MNIST generator on the copied RBT4DNN LoRA images and compares its generated images against the paper's per-requirement LoRA pass rates.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

for module in [
    "shared",
    "mnist",
    "precondition_validity",
    "cost_analysis",
    "mnist_lora_ablation",
    "mnist_shared_generator",
]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "mnist-shared-generator"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from mnist_shared_generator import TrainConfig, train_and_evaluate

paths = train_and_evaluate(ROOT, TrainConfig(epochs=100, samples_per_requirement=100))
for path in paths:
    print(path)